# Mühürlü test 2 (sealed2): v3 ve v4 karşılaştırması — tek sefer

`colab_v4.ipynb` üç seed'i bitirdikten sonra çalıştır. GPU gerekmez. **Tümünü çalıştır** → son hücrenin çıktısını yapıştır.
Bu test bir kez çalıştırılır; sonucuna göre ayar yapılmaz.


In [ ]:
import os, pathlib, shutil, subprocess, zipfile, json, torch
print("GPU:", torch.cuda.is_available(), "(gerekmez; yalnız tahmin yapılır)")
from google.colab import drive
drive.mount('/content/drive')
# Klasörler başka bir hesaptan paylaşılıp kısayol olarak eklenmiş olabilir: kısayol hedeflerinde de ara
ROOTS = [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/.shortcut-targets-by-id')]
def walk_dirs():
    for r in ROOTS:
        if r.exists():
            for d, subdirs, files in os.walk(r, followlinks=True):
                yield pathlib.Path(d), subdirs, files
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
if not DRIVE.exists():
    DRIVE = next((d / 'turkce-llm-dlp' for d, sub, _ in walk_dirs() if 'turkce-llm-dlp' in sub), DRIVE)
assert (DRIVE / 'runs').exists(), f'{DRIVE}/runs bulunamadı: turkce-llm-dlp klasörünü bu hesapta Drive\'ım\'a kısayol olarak ekle'
print('Drive klasörü:', DRIVE)
os.chdir('/content')
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
bundles = set()
for d, _, files in walk_dirs():
    if 'BUNDLE_INFO.txt' in files: bundles.add(d)
    bundles |= {d / f for f in files if f.startswith('turkce-dlp-colab-bundle') and f.endswith('.zip')}
for src in sorted(bundles, key=lambda p: p.name):
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp) if src.is_dir() else zipfile.ZipFile(src).extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
os.chdir('/content/proje')
assert pathlib.Path('data/ner/processed/augmented-v3/train.jsonl').exists(), 'augmented-v3 verisi yok (v5 paketi Drive\'da olmalı)'
!pip -q install -r training/ner/requirements.txt
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Mühürlü test 2 (sealed2, 100 toplanan mesaj): v3 ve v4 modelleri, üçer seed. BİR KEZ çalıştırılır.
for seed in (42, 43, 44):
    assert (DRIVE / 'runs' / f'augmented_v4_seed{seed}' / 'model').exists(), f'v4 seed {seed} henüz eğitilmedi (colab_v4.ipynb)'
!python training/ner/build_gold_set.py --sealed --sealed-dir sealed2 --out-dir data/ner/processed/sealed2-test > /dev/null
summary = {}
for run in ('augmented_v3', 'augmented_v4'):
    files = []
    for seed in (42, 43, 44):
        model = DRIVE / 'runs' / f'{run}_seed{seed}' / 'model'
        preds = f'/content/sealed2-{run}-{seed}.jsonl'
        !USE_TF=0 python training/ner/evaluate_model.py --model-dir "{model}" --data-dir data/ner/processed/sealed2-test --split test --dump-predictions {preds} > /dev/null
        target = DRIVE / 'runs' / f'{run}_seed{seed}' / 'sealed2-pipeline.json'
        !python training/ner/evaluate_rules.py --data-dir data/ner/processed/sealed2-test --splits test --with-predictions {preds} --output-json "{target}" > /dev/null
        files.append(str(target))
    out = subprocess.run(['python', 'training/ner/aggregate_seeds.py', *files], capture_output=True, text=True, check=True).stdout
    (DRIVE / 'runs' / f'sealed2-{run}-summary.json').write_text(out)
    m = json.loads(out)['metrics']
    for variant, label in (('pipeline_model_plus_rules', 'hibrit'), ('pipeline_model_only', 'yalnız model')):
        summary[f'{run} — {label}'] = {k.split('.')[-1]: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in m.items()
                                       if k.startswith(variant + '.') and any(x in k for x in ('_f1', 'overall_precision', 'overall_recall', '_coverage'))}
print(json.dumps(summary, ensure_ascii=False, indent=1))
